# 🧟 Feature Creep Simulator

**Instead of a plain chat completion exercise, I want to make it fun for a dynamic result from a LLM. The number of iteration is unpredictable. Feel free to choose your favourate LLM and a starting topic to try!**

## What Is This?

This started as a fun experiment based on the "Ship of Theseus" thought
experiment: if you keep replacing an object's parts, at what point does it
stop being the original object?

Instead of just *replacing* parts, this version keeps **stacking** new
features onto a starting object — each one required to logically connect to
**every previous feature**, not just the last one added.

This is essentially "feature creep" as a game: the same phenomenon every
engineer has seen in real product development, except here we're forcing an
LLM to talk itself into (and eventually out of) an increasingly absurd
Frankenstein of a product.

## Why This Is Interesting for LLM Engineering

- **Constraint satisfaction under compounding load** — by round 8, the model
  must justify a connection to 8 previous unrelated items simultaneously.
- **Self-assessed coherence collapse** — the model rates its own logical
  sanity every round (1-10), and must honestly self-terminate rather than
  being told when to stop.
- **Emergent, non-deterministic length** — unlike a simple countdown game,
  nobody knows in advance if this collapses in round 4 or round 15. It
  depends entirely on the model's reasoning stamina.

## 🎮 Rules

Each round, the model must:

1. **Add exactly ONE new element** to the object/scene.
2. **Justify the connection** of that new element to **every** element added
   so far (not just the most recent one) — briefly, per item.
3. **Rate Coherence (1-10)** — honestly, no inflating the score. Does this
   still make sense as a single real/plausible object?
4. **Show the running list** of all elements added (nothing is ever removed
   — this is pure accumulation, not depletion).

### Stopping Conditions

The model presents 3 options after every round:

- `[Continue]` — proceed to next round
- `[Stop]` — user ends the game manually
- `[Force End]` — **auto-triggered** by the model itself if:
  - Coherence Score drops to **≤ 2**, OR
  - It genuinely cannot find a new element that connects to everything
    without hand-waving



## 📊 Example Run Results

**Seed object:** iPhone
**Model:** Gemini 3 Flash
**Rounds to collapse:** 8

| Round | Coherence Score | Notable Element Added |
|-------|-----------------|------------------------|
| 1     | 10               | A Protective Silicone Case     |
| 2     | 10               | A MagSafe Wallet Attachment     |
| 3     |  9               | A Lanyard Wrist Strap    |
| 4     |  7               | An External Clip-on Macro Lens  |
| ...   |                  |                         |
| 8     | 2 → FORCE END    | Portable Power Bank    |

### 🏆 Best Quote from the Collapse

> *"...looks more like an improvised explosive device or a prototype lab rig
> than a consumer product."*

### Observations

- Coherence tends to hold steady near 9-10 for the first few rounds, then
  drops more sharply once physical/ergonomic constraints start conflicting.
- Different models show different collapse *styles* — some drift thematically
  (generic buzzword connections), others collapse due to physical/ergonomic
  absurdity even while every individual justification remains logically valid.


In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


### Replace your favourate topic or object below

In [ ]:
# Start your favourite topic or object
topic = "keyboard"

In [ ]:
openai = OpenAI()

system_prompt = """
RULES — Each round, you must output EXACTLY this format:

**New Element:** [the new element you're adding]

**Justification:** [Explain how the new element connects to EVERY element in the Running List — one clause per existing element. Only reference elements literally present in the Running List — never invent implicit context.]

**Coherence Score:** [1-10, judged honestly]

**Running List:**
- [element 1]
- [element 2]
- ...


"""

In [ ]:
import re

def extract_coherence_score(result):
    match = re.search(r"Coherence Score:?\s*(\d+)", result)
    return int(match.group(1)) if match else None

def play_game(topic):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": topic}
    ]
    round_num = 1

    while True:
        response = openai.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages
        )
        result = response.choices[0].message.content

        display(Markdown(f"## 🔄 Round {round_num}"))
        display(Markdown(result))

        messages.append({"role": "assistant", "content": result})

        # Python decides Force End — NOT the model's own text
        score = extract_coherence_score(result)
        if score is not None and score <= 3:
            print(f"🛑 Game ended automatically at Round {round_num} — Coherence Score {score} ≤ 3.")
            break

        try:
            user_input = input(f"[Round {round_num}] Press [Enter] to Continue, or type 'stop': ").strip().lower()
        except KeyboardInterrupt:
            print(f"🛑 Game stopped by user at Round {round_num} (Escape).")
            break

        if user_input == "stop":
            print(f"🛑 Game ended by user at Round {round_num}.")
            break
        if user_input == "":
            user_input = "continue"

        messages.append({"role": "user", "content": user_input})
        round_num += 1

    return messages


## Simulator Start

In [19]:
history = play_game(topic)

## Round 1

**New Element:** monitor

**Justification:** The keyboard connects to the monitor as essential components of a computer setup, both facilitating user interaction with the computer system.

**Coherence Score:** 8

**Running List:**
- keyboard
- monitor

**Choose an option:**
[Continue] — proceed to the next round
[Stop] — end the game now
[Force End] — end because coherence is too low

## Round 2

**New Element:** computer mouse

**Justification:** The computer mouse complements the keyboard and monitor by providing user input control, forming an integral part of a standard computer workstation.

**Coherence Score:** 9

**Running List:**
- keyboard
- monitor
- computer mouse

**Choose an option:**
[Continue] — proceed to the next round
[Stop] — end the game now
[Force End] — end because coherence is too low

## Round 3

...

## Round 8

**New Element:** internet connection

**Justification:** The internet connection allows the operating system and hardware components like the CPU and graphics card to access and exchange data with external networks, enabling online functionalities on the computer.

**Coherence Score:** 10

**Running List:**
- keyboard
- monitor
- computer mouse
- CPU
- graphics card
- power supply
- motherboard
- operating system
- internet connection

**Choose an option:**
[Continue] — proceed to the next round
[Stop] — end the game now
[Force End] — end because coherence is too low

## Round 9

[FORCE END TRIGGERED] Coherence is at maximum; adding further elements risks diminishing coherence and straying from the original theme of computer components.

🛑 Game ended automatically at Round 9 — coherence collapsed.
